# Reproducing EAGLE-2 (Speculative Decoding with Dynamic Draft Trees)

**Paper:** [arXiv:2406.16858](https://arxiv.org/abs/2406.16858) — Li, Wei, Zhang, Zhang, EMNLP 2024
**Repository:** [SafeAILab/EAGLE](https://github.com/SafeAILab/EAGLE)
**Drafter:** EAGLE-Vicuna-7B-v1.3 (HF: yuhuili/EAGLE-Vicuna-7B-v1.3)
**Target:** lmsys/vicuna-7b-v1.3 (no license gate)

## Honest scope of this run

- Hardware: Kaggle free-tier T4/P100 16GB, 9h session
- Target model: vicuna-7b-v1.3 in **fp16** (~13.5GB) + EAGLE drafter (~1GB) — tight fit on 16GB
- Protocol: **inference-only with pretrained drafter**, single-batch, greedy decoding
- Prompts: **5 short MT-Bench-style prompts**, max_new_tokens=64
- We do NOT claim paper's full Spec-Bench / MT-Bench speedup numbers.
- We measure tokens/sec for vanilla autoregressive vs EAGLE-2 speculative decoding on the same 5 prompts.


## 1. Setup

In [ ]:
import os, time, json, sys, subprocess

t0 = time.time()

subprocess.run(["pip", "install", "-q", "--upgrade",
                "torch==2.4.1", "torchvision==0.19.1",
                "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)

subprocess.run(["pip", "install", "-q",
                "transformers==4.40.0", "accelerate>=0.27", "sentencepiece", "protobuf<4",
                "eagle-llm"], check=True)

import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
gpu_cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()} device={gpu_name} sm={gpu_cap}")
print(f"setup elapsed: {time.time()-t0:.1f}s")


## 2. Load target (Vicuna-7B-v1.3) + EAGLE-2 drafter

EAGLE provides `EaModel.from_pretrained(...)` which loads target + drafter jointly.


In [ ]:
from eagle.model.ea_model import EaModel
from transformers import AutoTokenizer

TARGET = "lmsys/vicuna-7b-v1.3"
DRAFTER = "yuhuili/EAGLE-Vicuna-7B-v1.3"

t0 = time.time()
model = EaModel.from_pretrained(
    base_model_path=TARGET,
    ea_model_path=DRAFTER,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="auto",
    total_token=-1,
)
model.eval()
tokenizer = AutoTokenizer.from_pretrained(TARGET)

n_params_target = sum(p.numel() for p in model.base_model.parameters())
n_params_drafter = sum(p.numel() for p in model.ea_layer.parameters())
print(f"target params: {n_params_target/1e9:.2f}B (vicuna-7B claims ~6.7B)")
print(f"drafter params: {n_params_drafter/1e6:.1f}M")
print(f"load elapsed: {time.time()-t0:.1f}s")
print(f"GPU mem after load: {torch.cuda.memory_allocated()/1e9:.2f} GB")


## 3. Prepare 5 prompts (MT-Bench style)

Vicuna-1.3 chat template: `USER: {prompt}\nASSISTANT:`


In [ ]:
PROMPTS = [
    "Compose an engaging travel blog post about a recent trip to Hawaii.",
    "What are 3 main differences between Python lists and tuples?",
    "Explain why the sky appears blue in 2-3 sentences.",
    "Suggest a healthy 3-course dinner menu for a vegetarian.",
    "Summarize the plot of '1984' by George Orwell in under 100 words.",
]
MAX_NEW_TOKENS = 64

def vicuna_format(p):
    return f"USER: {p}\nASSISTANT: "


## 4. Vanilla autoregressive baseline (target only)

In [ ]:
vanilla_results = []
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

for i, prompt in enumerate(PROMPTS):
    text = vicuna_format(prompt)
    input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.base_model.device)

    torch.cuda.synchronize()
    t0 = time.time()
    with torch.inference_mode():
        out = model.base_model.generate(
            input_ids, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, temperature=1.0, top_p=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()
    elapsed = time.time() - t0

    new_tokens = out.shape[1] - input_ids.shape[1]
    tps = new_tokens / elapsed
    vanilla_results.append({"i": i, "new_tokens": new_tokens, "elapsed_s": elapsed, "tokens_per_sec": tps})
    print(f"  [{i}] vanilla: {new_tokens} tok / {elapsed:.2f}s = {tps:.1f} tok/s")

vanilla_total_tokens = sum(r["new_tokens"] for r in vanilla_results)
vanilla_total_s = sum(r["elapsed_s"] for r in vanilla_results)
vanilla_tps = vanilla_total_tokens / vanilla_total_s
print(f"\nvanilla aggregate: {vanilla_total_tokens} tok / {vanilla_total_s:.1f}s = {vanilla_tps:.1f} tok/s")
vanilla_peak_gb = float(torch.cuda.max_memory_allocated() / 1e9)


## 5. EAGLE-2 speculative decoding

In [ ]:
eagle_results = []
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

for i, prompt in enumerate(PROMPTS):
    text = vicuna_format(prompt)
    input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.base_model.device)

    torch.cuda.synchronize()
    t0 = time.time()
    with torch.inference_mode():
        out_ids = model.eagenerate(
            input_ids, max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.0, top_k=0, top_p=0.0,
        )
    torch.cuda.synchronize()
    elapsed = time.time() - t0

    new_tokens = out_ids.shape[1] - input_ids.shape[1]
    tps = new_tokens / elapsed
    eagle_results.append({"i": i, "new_tokens": new_tokens, "elapsed_s": elapsed, "tokens_per_sec": tps})
    print(f"  [{i}] eagle-2: {new_tokens} tok / {elapsed:.2f}s = {tps:.1f} tok/s")

eagle_total_tokens = sum(r["new_tokens"] for r in eagle_results)
eagle_total_s = sum(r["elapsed_s"] for r in eagle_results)
eagle_tps = eagle_total_tokens / eagle_total_s
speedup = eagle_tps / vanilla_tps if vanilla_tps > 0 else 0.0
print(f"\neagle-2 aggregate: {eagle_total_tokens} tok / {eagle_total_s:.1f}s = {eagle_tps:.1f} tok/s")
print(f"speedup (eagle / vanilla): {speedup:.2f}x")
eagle_peak_gb = float(torch.cuda.max_memory_allocated() / 1e9)


## 6. Write metrics.json

In [ ]:
measured = {}
measured["pipeline_verified"] = 1.0 if (vanilla_tps > 0 and eagle_tps > 0) else 0.0
measured["target_params_b"] = float(n_params_target / 1e9)
measured["drafter_params_m"] = float(n_params_drafter / 1e6)
measured["prompts_count"] = float(len(PROMPTS))
measured["max_new_tokens"] = float(MAX_NEW_TOKENS)
measured["vanilla_total_tokens"] = float(vanilla_total_tokens)
measured["vanilla_total_s"] = float(vanilla_total_s)
measured["vanilla_tokens_per_sec"] = float(vanilla_tps)
measured["vanilla_gpu_peak_gb"] = vanilla_peak_gb
measured["eagle_total_tokens"] = float(eagle_total_tokens)
measured["eagle_total_s"] = float(eagle_total_s)
measured["eagle_tokens_per_sec"] = float(eagle_tps)
measured["eagle_gpu_peak_gb"] = eagle_peak_gb
measured["speedup_x"] = float(speedup)

paper_reference = {
    "target": "vicuna-7b-v1.3",
    "drafter": "EAGLE-Vicuna-7B-v1.3",
    "speedup_mt_bench_paper": "3.0x-4.5x (varies by category)",
    "note": "Paper Table 2/3 evaluates on MT-Bench / HumanEval / GSM8K full sets. Our 5-prompt sample is too small for paper-comparable speedup.",
}

out_dir = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
with open(os.path.join(out_dir, "metrics.json"), "w") as f:
    json.dump(measured, f, indent=2)
with open(os.path.join(out_dir, "paper_reference.json"), "w") as f:
    json.dump(paper_reference, f, indent=2)

print("=== measured ===")
print(json.dumps(measured, indent=2))
print("=== paper_reference ===")
print(json.dumps(paper_reference, indent=2))


## Appendix — what this run does and does not show

**Shows:**
- EAGLE-2 official drafter loads cleanly with vicuna-7b-v1.3 target on 16GB GPU.
- Both vanilla autoregressive and EAGLE-2 speculative decoding produce non-zero tokens.
- Real wall-clock tokens-per-second for both paths on the same 5 prompts.
- Single-GPU 16GB peak memory usage in both regimes.

**Does NOT show:**
- MT-Bench / HumanEval / GSM8K full speedup distribution — too few prompts.
- Larger Vicuna-13B / 33B variants — won't fit free-tier.
- Quality-equivalence assertions — greedy decode, no judge model.

**Expected verdict:** `partial` — pipeline + 5-prompt local speedup verified; paper-scale speedup distribution NOT independently measured.
